# NEON SPACE SURVIVOR — Game Logic Notebook

This notebook explores the rules used by the Streamlit arcade game: sector generation, movement, collectibles, asteroid collisions, and scoring. Run cells from top to bottom.

In [ ]:
import random
GRID = 7
START = (3, 6)
INITIAL_LIVES = 3
ORB_POINTS = 25


## 1. Generate a mission sector

In [ ]:
def create_sector(seed=7):
    rng = random.Random(seed)
    hazards = set()
    while len(hazards) < 7:
        p = (rng.randrange(GRID), rng.randrange(GRID))
        if p != START:
            hazards.add(p)
    orbs = set()
    while len(orbs) < 5:
        p = (rng.randrange(GRID), rng.randrange(GRID))
        if p not in hazards and p != START:
            orbs.add(p)
    return hazards, orbs

hazards, orbs = create_sector()
print('Asteroids:', sorted(hazards))
print('Energy orbs:', sorted(orbs))

## 2. Render the board

In [ ]:
def render_board(player, hazards, orbs):
    for r in range(GRID):
        row = []
        for c in range(GRID):
            p = (r, c)
            row.append('🚀' if p == player else '☄️' if p in hazards else '💠' if p in orbs else '·')
        print(' '.join(row))

render_board(START, hazards, orbs)

## 3. Apply a turn

In [ ]:
def take_turn(player, direction, hazards, orbs, score=0, lives=INITIAL_LIVES):
    offsets = {'up':(-1,0), 'down':(1,0), 'left':(0,-1), 'right':(0,1)}
    if direction not in offsets:
        raise ValueError('Direction must be up, down, left, or right')
    dr, dc = offsets[direction]
    pos = (player[0] + dr, player[1] + dc)
    if not (0 <= pos[0] < GRID and 0 <= pos[1] < GRID):
        return player, hazards, orbs, score, lives, 'Boundary reached'
    score += 2
    if pos in hazards:
        hazards = set(hazards); hazards.remove(pos); lives -= 1
        event = 'Asteroid collision: one shield lost'
    elif pos in orbs:
        orbs = set(orbs); orbs.remove(pos); score += ORB_POINTS
        event = 'Energy orb collected'
    else:
        event = 'Safe move'
    return pos, hazards, orbs, score, lives, event


## 4. Run a sample turn

In [ ]:
player = START
score, lives = 0, INITIAL_LIVES
player, hazards, orbs, score, lives, event = take_turn(player, 'up', hazards, orbs, score, lives)
print(event, '| score:', score, '| shields:', lives, '| orbs left:', len(orbs))
render_board(player, hazards, orbs)

## Game objective

Collect every energy orb to win. The mission ends when the shield count reaches zero. The Streamlit app wraps these rules in an interactive interface with session-state tracking and restart controls.